# 07 - Silver to Gold (Nordhaus O2C)

Builds the Kimball star that the Direct Lake semantic model and the Ontology sit on.

## Two Fabric Graph constraints that drive every design decision here

1. **No `Decimal` anywhere in Gold.** Fabric Graph returns `null` for every Decimal property.
   Silver deliberately keeps Decimal (it is the honest type for SAP CURR/QUAN); the cast to
   `Double` belongs here and nowhere else.
2. **No spaces or special characters in Gold column names.** Anything outside
   `^[A-Za-z][A-Za-z0-9_]*$` makes Delta enable column mapping, which the graph cannot read.

Both are enforced mechanically in section 9 - the notebook refuses to write a table that
violates either. They are too easy to reintroduce by hand to be left as documentation.

## Load semantics differ by table type, on purpose

| | Mode | Why |
|---|---|---|
| SCD2 dimensions | **merge** | overwriting would destroy version history on every re-run |
| Type-1 dimensions, facts | **overwrite** | fully derivable from Silver, so rebuilding is safe and idempotent |

With a single source snapshot the SCD2 dimensions produce one open version per key. The merge
is written properly regardless, so the second load of a changed attribute closes the old row and
opens a new one. Claiming history that the source does not contain would be fiction; building the
mechanism that will capture it is not.

## Grain

`FactOrderToCash` is an **accumulating snapshot at sales-order-line grain** - one row per order
line, carrying every O2C milestone date and the lags between them. That is the shape that lets a
single fact answer "where does the process stall", which a transaction-grain fact cannot.


In [ ]:
from pyspark.sql import functions as F, Window as W
from pyspark.sql import types as T
from delta.tables import DeltaTable
import re

SILVER = "slv_"
GOLD   = "gld_"
HIGH_DATE = "9999-12-31"

def dbl(c):
    """Gold rule 1: every numeric lands as Double. Fabric Graph cannot read Decimal."""
    return F.col(c).cast("double")

def datekey(c):
    """Surrogate date key as yyyyMMdd integer; NULL stays NULL rather than becoming 0."""
    return F.when(F.col(c).isNull(), None).otherwise(F.date_format(F.col(c), "yyyyMMdd").cast("int"))

print("helpers ready")

## 1. DimDate

In [ ]:
ful = spark.table(f"{SILVER}order_fulfilment")
ar  = spark.table(f"{SILVER}ar_item")

_b = ful.select(
        F.min("OrderCreatedDate").alias("lo1"), F.max("LastBillingDate").alias("hi1"),
        F.max("RequestedDeliveryDate").alias("hi2"), F.max("LastGoodsIssueDate").alias("hi3")
     ).collect()[0]
_a = ar.select(F.min("PostingDate").alias("lo2"),
               F.max("DueDate").alias("hi4"), F.max("ClearingDate").alias("hi5")).collect()[0]

_los = [d for d in [_b["lo1"], _a["lo2"]] if d is not None]
_his = [d for d in [_b["hi1"], _b["hi2"], _b["hi3"], _a["hi4"], _a["hi5"]] if d is not None]
start = f"{min(_los).year}-01-01"
end   = f"{max(_his).year}-12-31"
print("calendar span:", start, "->", end)

gld_date = (spark.sql(f"SELECT explode(sequence(to_date('{start}'), to_date('{end}'), interval 1 day)) AS Date")
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("QuarterName", F.concat(F.lit("Q"), F.quarter("Date")))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("MonthShort", F.date_format("Date", "MMM"))
    .withColumn("YearMonth", F.date_format("Date", "yyyyMM"))
    .withColumn("YearMonthName", F.date_format("Date", "yyyy-MM"))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
    .withColumn("DayOfWeek", F.dayofweek("Date"))
    .withColumn("DayName", F.date_format("Date", "EEEE"))
    .withColumn("WeekOfYear", F.weekofyear("Date"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin(1, 7))
    .select("DateKey", "Date", "Year", "Quarter", "QuarterName", "MonthNumber", "MonthName",
            "MonthShort", "YearMonth", "YearMonthName", "DayOfMonth", "DayOfWeek", "DayName",
            "WeekOfYear", "IsWeekend"))

print("DimDate rows:", gld_date.count())

## 2. SCD2 dimensions

`RowHash` over the tracked attributes is what decides whether a row has actually changed. Hashing
the attributes rather than comparing column by column means adding an attribute later cannot
silently stop change detection.

In [ ]:
def scd2_load(src, table, business_key, attrs, load_date):
    """Type-2 load. Seeds on first run; on later runs closes changed rows and opens new versions.

    Deliberately a merge, not an overwrite: an overwrite would silently destroy history every
    time the notebook re-runs, which is the most common way SCD2 implementations turn out to be
    SCD1 wearing a costume.
    """
    staged = (src
        .withColumn("RowHash", F.sha2(F.concat_ws("||", *[F.coalesce(F.col(a).cast("string"), F.lit("~")) for a in attrs]), 256))
        .withColumn("ValidFrom", F.lit(load_date).cast("date"))
        .withColumn("ValidTo", F.lit(HIGH_DATE).cast("date"))
        .withColumn("IsCurrent", F.lit(True))
        .withColumn("VersionNumber", F.lit(1)))

    full = f"{GOLD}{table}"
    if not spark.catalog.tableExists(full):
        staged.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full)
        print(f"{full}: seeded {staged.count():,} rows")
        return spark.table(full)

    tgt = DeltaTable.forName(spark, full)
    cur = spark.table(full).filter("IsCurrent")
    on_sc = [F.col(f"s.{k}") == F.col(f"c.{k}") for k in business_key]
    changed = (staged.alias("s").join(cur.alias("c"), on_sc, "inner")
                     .filter(F.col("s.RowHash") != F.col("c.RowHash"))
                     .select(*[F.col(f"s.{k}").alias(k) for k in business_key]))
    n_changed = changed.count()

    if n_changed:
        (tgt.alias("t").merge(
            changed.alias("x"),
            " AND ".join([f"t.{k} = x.{k}" for k in business_key]) + " AND t.IsCurrent = true")
           .whenMatchedUpdate(set={"IsCurrent": F.lit(False),
                                   "ValidTo": F.lit(load_date).cast("date")})
           .execute())

    existing_hashes = spark.table(full).select(*business_key, "RowHash", "VersionNumber")
    on_se = [F.col(f"s.{k}") == F.col(f"e.{k}") for k in business_key]
    new_rows = (staged.alias("s")
        .join(existing_hashes.alias("e"), on_se, "left")
        .filter(F.col("e.RowHash").isNull() | (F.col("e.RowHash") != F.col("s.RowHash")))
        .select("s.*"))
    n_new = new_rows.count()
    if n_new:
        new_rows.write.format("delta").mode("append").saveAsTable(full)

    print(f"{full}: {n_changed:,} versions closed, {n_new:,} rows appended")
    return spark.table(full)


LOAD_DATE = spark.sql("select current_date() as d").collect()[0]["d"].isoformat()
print("load date:", LOAD_DATE)

In [ ]:
cust_src = (spark.table(f"{SILVER}customer").alias("c")
    .join(spark.table(f"{SILVER}customer_sales_area")
            .groupBy("CustomerId")
            .agg(F.first("CustomerGroup", True).alias("CustomerGroup"),
                 F.first("SalesDistrict", True).alias("SalesDistrict"),
                 F.first("PriceGroup", True).alias("PriceGroup"),
                 F.first("PaymentTerms", True).alias("PaymentTerms"),
                 F.first("Incoterms", True).alias("Incoterms")).alias("sa"),
          "CustomerId", "left")
    .select("CustomerId", "CustomerName", "CountryKey", "City", "Region", "PostalCode",
            "AccountGroup", "IndustryKey", "CustomerGroup", "SalesDistrict", "PriceGroup",
            "PaymentTerms", "Incoterms", "IsDeleted"))

CUST_ATTRS = ["CustomerName", "CountryKey", "City", "Region", "AccountGroup", "IndustryKey",
              "CustomerGroup", "SalesDistrict", "PriceGroup", "PaymentTerms", "Incoterms", "IsDeleted"]

mat_src = (spark.table(f"{SILVER}material")
    .select("MaterialId", "MaterialName", "MaterialType", "MaterialGroup", "BaseUnit",
            F.col("GrossWeight").cast("double").alias("GrossWeight"),
            F.col("NetWeight").cast("double").alias("NetWeight"),
            "WeightUnit", "ProductHierarchy", "IsDeleted"))

MAT_ATTRS = ["MaterialName", "MaterialType", "MaterialGroup", "BaseUnit",
             "GrossWeight", "NetWeight", "ProductHierarchy", "IsDeleted"]

gld_customer = scd2_load(cust_src, "dim_customer", ["CustomerId"], CUST_ATTRS, LOAD_DATE)
gld_material = scd2_load(mat_src,  "dim_material", ["MaterialId"], MAT_ATTRS,  LOAD_DATE)

## 3. Type-1 dimensions

In [ ]:
gld_salesorg = (spark.table(f"{SILVER}sales_org").alias("s")
    .join(spark.table(f"{SILVER}company").alias("c"), "CompanyCode", "left")
    .select("SalesOrg", "SalesOrgName", "CompanyCode", "CompanyName",
            F.col("c.CountryKey").alias("CountryKey"), "LocalCurrency"))

gld_plant     = spark.table(f"{SILVER}plant").select("Plant", "PlantName", "CountryKey", "City", "SalesOrg")
gld_division  = spark.table(f"{SILVER}division").select("Division", "DivisionName")
gld_channel   = spark.table(f"{SILVER}distribution_channel").select("DistributionChannel", "ChannelName")
gld_ordertype = spark.table(f"{SILVER}order_type").select("OrderType", "OrderTypeText")
gld_rejection = spark.table(f"{SILVER}rejection_reason").select("RejectionReason", "RejectionReasonText")

for n, d in [("sales_org", gld_salesorg), ("plant", gld_plant), ("division", gld_division),
             ("channel", gld_channel), ("order_type", gld_ordertype), ("rejection", gld_rejection)]:
    print(f"{n:12s}{d.count():>6,}")

## 4. Currency conversion

Nordhaus bills in EUR, USD and SGD. Without translation the exec scoreboard silently adds three
currencies together, which is worse than showing nothing. Rates come from `ref_fx_rates`, loaded
by the Dataflow Gen2, joined on the **order month** so a line is valued at the rate that applied
when it was taken.

In [ ]:
fx = (spark.table("ref_fx_rates")
        .select(F.col("YEARMONTH").alias("FxYearMonth"),
                F.col("FROM_CURR").alias("FxCurrency"),
                F.col("RATE").cast("double").alias("FxRate")))

print("fx rows:", fx.count())
fx.groupBy("FxCurrency").agg(F.min("FxRate").alias("min"), F.max("FxRate").alias("max")).show()

## 5. FactOrderToCash - the accumulating snapshot

One row per order line. Milestones run order -> requested -> confirmed -> goods issue -> invoice
-> cash, and the lag columns between them are what make bottleneck analysis a single-table query.

The cash milestone is reached by walking order line -> billing item -> accounting document ->
AR item, aggregated back to the line so the join cannot fan out the grain.

In [ ]:
bil = spark.table(f"{SILVER}billing_item")
arx = spark.table(f"{SILVER}ar_item")

# order line -> accounting doc -> AR clearing, collapsed back to line grain
ar_by_line = (bil.filter(F.col("SalesOrderId").isNotNull() & F.col("AccountingDocId").isNotNull())
    .select("SalesOrderId", "SalesOrderItem", "AccountingDocId").dropDuplicates()
    .join(arx.select("AccountingDocId", "ClearingDate", "DueDate", "IsOpen", "DaysToPay",
                     "DaysOverdue", "SignedAmountLocal"), "AccountingDocId", "inner")
    .groupBy("SalesOrderId", "SalesOrderItem")
    .agg(F.max("ClearingDate").alias("LastClearingDate"),
         F.min("DueDate").alias("FirstDueDate"),
         F.max(F.col("IsOpen").cast("int")).alias("_anyOpen"),
         F.avg(F.col("DaysToPay").cast("double")).alias("AvgDaysToPay"),
         F.max(F.col("DaysOverdue").cast("double")).alias("MaxDaysOverdue"),
         F.sum(F.col("SignedAmountLocal").cast("double")).alias("ArAmountLocal"))
    .withColumn("IsFullyCleared", F.col("_anyOpen") == 0)
    .drop("_anyOpen"))

f = (ful.alias("f")
     .join(ar_by_line, ["SalesOrderId", "SalesOrderItem"], "left")
     .withColumn("OrderYearMonth", F.date_format("OrderCreatedDate", "yyyyMM"))
     .join(fx, (F.col("OrderYearMonth") == F.col("FxYearMonth")) & (F.col("DocCurrency") == F.col("FxCurrency")), "left")
     .withColumn("FxRateMissing", F.col("FxRate").isNull() & (F.col("DocCurrency") != "EUR"))
     .withColumn("FxRate", F.coalesce(F.col("FxRate"), F.lit(1.0))))

gld_fact = (f
    # --- degenerate + foreign keys -------------------------------------------------
    .withColumn("OrderLineKey", F.concat_ws("-", "SalesOrderId", "SalesOrderItem"))
    .withColumn("OrderCreatedDateKey", datekey("OrderCreatedDate"))
    .withColumn("RequestedDeliveryDateKey", datekey("RequestedDeliveryDate"))
    .withColumn("ConfirmedDeliveryDateKey", datekey("ConfirmedDeliveryDate"))
    .withColumn("FirstGoodsIssueDateKey", datekey("FirstGoodsIssueDate"))
    .withColumn("LastGoodsIssueDateKey", datekey("LastGoodsIssueDate"))
    .withColumn("FirstBillingDateKey", datekey("FirstBillingDate"))
    .withColumn("LastBillingDateKey", datekey("LastBillingDate"))
    .withColumn("ClearingDateKey", datekey("LastClearingDate"))
    # --- measures, all Double ------------------------------------------------------
    .withColumn("OrderQty", dbl("OrderQty"))
    .withColumn("DeliveredQty", dbl("DeliveredQty"))
    .withColumn("BilledQty", dbl("BilledQty"))
    .withColumn("OrderNetValue", dbl("ItemNetValue"))
    .withColumn("ListValue", dbl("ListValue"))
    .withColumn("ItemCost", dbl("ItemCost"))
    .withColumn("DiscountAmount", dbl("DiscountAmount"))
    .withColumn("DiscountPct", dbl("DiscountPct"))
    .withColumn("BilledNetValue", dbl("BilledNetValue"))
    # Rounded to 2dp on purpose. The Decimal -> Double cast is forced by Fabric Graph, and its
    # cost is IEEE-754 noise (5189.200082400001 rather than 5189.20), which is visible in the
    # ontology instance browser. Rounding keeps the type Double and removes the artefact.
    .withColumn("OrderNetValueEUR", F.round(dbl("ItemNetValue") * F.col("FxRate"), 2))
    .withColumn("BilledNetValueEUR", F.round(dbl("BilledNetValue") * F.col("FxRate"), 2))
    .withColumn("ItemCostEUR", F.round(dbl("ItemCost") * F.col("FxRate"), 2))
    .withColumn("MarginEUR", F.round((dbl("ItemNetValue") - dbl("ItemCost")) * F.col("FxRate"), 2))
    .withColumn("ArAmountLocal", dbl("ArAmountLocal"))
    # --- process lags --------------------------------------------------------------
    .withColumn("OrderToGoodsIssueDays", F.col("OrderToGoodsIssueDays").cast("int"))
    .withColumn("GoodsIssueToInvoiceDays", F.col("GoodsIssueToInvoiceDays").cast("int"))
    .withColumn("OrderToInvoiceDays", F.col("OrderToInvoiceDays").cast("int"))
    .withColumn("InvoiceToCashDays", F.datediff("LastClearingDate", "LastBillingDate").cast("int"))
    .withColumn("OrderToCashDays", F.datediff("LastClearingDate", "OrderCreatedDate").cast("int"))
    .withColumn("DeliveryDelayDays", F.col("DeliveryDelayDays").cast("int"))
    .withColumn("AvgDaysToPay", dbl("AvgDaysToPay"))
    .withColumn("MaxDaysOverdue", dbl("MaxDaysOverdue"))
    .withColumn("DeliveryCount", F.col("DeliveryCount").cast("int"))
    .withColumn("BillingDocCount", F.coalesce(F.col("BillingDocCount"), F.lit(0)).cast("int"))
    .withColumn("IsFullyCleared", F.coalesce(F.col("IsFullyCleared"), F.lit(False)))
    .select(
        "OrderLineKey", "SalesOrderId", "SalesOrderItem",
        "SoldToId", "ShipToId", "MaterialId", "Plant", "SalesOrg", "DistributionChannel",
        "Division", "OrderType", "MaterialGroup", "RejectionReason", "DocCurrency", "FxRate", "FxRateMissing",
        "OrderCreatedDate", "RequestedDeliveryDate", "ConfirmedDeliveryDate",
        "FirstGoodsIssueDate", "LastGoodsIssueDate", "FirstBillingDate", "LastBillingDate",
        "LastClearingDate", "FirstDueDate",
        "OrderCreatedDateKey", "RequestedDeliveryDateKey", "ConfirmedDeliveryDateKey",
        "FirstGoodsIssueDateKey", "LastGoodsIssueDateKey", "FirstBillingDateKey",
        "LastBillingDateKey", "ClearingDateKey",
        "OrderQty", "DeliveredQty", "BilledQty",
        "OrderNetValue", "ListValue", "ItemCost", "DiscountAmount", "DiscountPct",
        "BilledNetValue", "OrderNetValueEUR", "BilledNetValueEUR", "ItemCostEUR", "MarginEUR",
        "ArAmountLocal", "AvgDaysToPay", "MaxDaysOverdue",
        "OrderToGoodsIssueDays", "GoodsIssueToInvoiceDays", "OrderToInvoiceDays",
        "InvoiceToCashDays", "OrderToCashDays", "DeliveryDelayDays",
        "DeliveryCount", "BillingDocCount",
        "IsRejected", "IsDelivered", "IsInFull", "IsOnTime", "IsOnTimeVsConfirmed", "IsOTIF",
        "IsSplitDelivery", "IsOpenBacklog", "IsFullyCleared",
        "HasDeliveryBlock", "HasBillingBlock", "CreditStatus", "ShipsElsewhere"))

print("FactOrderToCash rows:", gld_fact.count())

## 6. FactAccountsReceivable and FactSalesTarget

In [ ]:
# AR amounts are SAP DMBTR - the COMPANY CODE's local currency, not group currency.
# DE10 is EUR, US10 is USD, SG10 is SGD, so summing AmountLocal across company codes adds
# three currencies together. That is precisely the error the FX conversion on the order fact
# exists to prevent, so AR gets the same treatment: converted at the rate for its POSTING
# month, which is when the receivable came into existence.
fx_ar = (spark.table("ref_fx_rates")
         .select(F.col("YEARMONTH").alias("ArFxYearMonth"),
                 F.col("FROM_CURR").alias("ArFxCurrency"),
                 F.col("RATE").cast("double").alias("ArFxRateRaw")))

gld_ar = (arx
    .withColumn("PostingYearMonth", F.date_format("PostingDate", "yyyyMM"))
    .join(fx_ar,
          (F.col("PostingYearMonth") == F.col("ArFxYearMonth"))
          & (F.col("Currency") == F.col("ArFxCurrency")), "left")
    # A missing rate must be loud. Silently defaulting to 1.0 would misstate every USD and
    # SGD receivable while looking perfectly healthy.
    .withColumn("FxRateMissing",
                F.col("ArFxRateRaw").isNull() & (F.trim(F.col("Currency")) != "EUR"))
    .withColumn("FxRate", F.coalesce(F.col("ArFxRateRaw"), F.lit(1.0)))
    .withColumn("PostingDateKey", datekey("PostingDate"))
    .withColumn("BaselineDateKey", datekey("BaselineDate"))
    .withColumn("DueDateKey", datekey("DueDate"))
    .withColumn("ClearingDateKey", datekey("ClearingDate"))
    .withColumn("AmountLocal", dbl("AmountLocal"))
    .withColumn("AmountDoc", dbl("AmountDoc"))
    .withColumn("SignedAmountLocal", dbl("SignedAmountLocal"))
    .withColumn("AmountEUR", F.round(dbl("AmountLocal") * F.col("FxRate"), 2))
    .withColumn("SignedAmountEUR", F.round(dbl("SignedAmountLocal") * F.col("FxRate"), 2))
    .withColumn("DaysToPay", F.col("DaysToPay").cast("int"))
    .withColumn("DaysOverdue", F.col("DaysOverdue").cast("int"))
    .withColumn("CashDiscountDays", dbl("CashDiscountDays"))
    .select("CompanyCode", "CustomerId", "AccountingDocId", "FiscalYear", "LineItem",
            "PostingDate", "BaselineDate", "DueDate", "ClearingDate",
            "PostingDateKey", "BaselineDateKey", "DueDateKey", "ClearingDateKey",
            "Currency", "FxRate", "FxRateMissing",
            "BillingDocRef", "DocumentType", "DebitCreditInd", "PaymentTerms",
            "AmountLocal", "AmountDoc", "SignedAmountLocal",
            "AmountEUR", "SignedAmountEUR",
            "DaysToPay", "DaysOverdue", "CashDiscountDays",
            "IsOpen", "IsOverdue", "SourceTable"))

gld_target = (spark.table("ref_sales_target")
    .select(F.col("YEARMONTH").alias("YearMonth"),
            F.col("VKORG").alias("SalesOrg"),
            F.col("SPART").alias("Division"),
            F.col("TARGET_NET_EUR").cast("double").alias("TargetNetEUR"),
            F.col("CURRENCY").alias("Currency"))
    .withColumn("MonthStartDateKey",
                (F.col("YearMonth").cast("int") * 100 + 1).cast("int")))

gld_docflow = (spark.table(f"{SILVER}doc_flow")
    .withColumn("ReferencedQty", dbl("ReferencedQty"))
    .withColumn("ReferencedValue", dbl("ReferencedValue"))
    .withColumn("FlowCreatedDateKey", datekey("FlowCreatedDate"))
    # Bridge keys for the ontology. Entity relationships target the entity KEY, and
    # gld_fact_order_to_cash is keyed on OrderLineKey (SalesOrderId-SalesOrderItem). The flow
    # table carries the two parts but not the concatenation, so it is built here.
    # The category guard is deliberate: only 'C' (sales order) rows point at an order line, so
    # the key stays NULL for delivery / invoice / credit-memo rows rather than becoming a
    # same-shaped string that joins to nothing and looks like a data defect.
    .withColumn("PrecedingOrderLineKey",
                F.when(F.col("PrecedingCategory") == "C",
                       F.concat_ws("-", "PrecedingDocId", "PrecedingItem")))
    .withColumn("SubsequentOrderLineKey",
                F.when(F.col("SubsequentCategory") == "C",
                       F.concat_ws("-", "SubsequentDocId", "SubsequentItem")))
    .select("PrecedingDocId", "PrecedingItem", "PrecedingType", "PrecedingCategory",
            "PrecedingOrderLineKey",
            "SubsequentDocId", "SubsequentItem", "SubsequentType", "SubsequentCategory",
            "SubsequentOrderLineKey",
            "ReferencedQty", "ReferencedValue", "FlowCreatedDate", "FlowCreatedDateKey"))

print("AR:", gld_ar.count(), "| targets:", gld_target.count(), "| flow edges:", gld_docflow.count())

## 7. Gold constraint gate

Runs **before** anything is written. Both rules below have already cost this project once each in
other forms, and neither is visible at write time - a Decimal column writes perfectly happily and
only returns null much later, inside the graph.

In [ ]:
NAME_RE = re.compile(r"^[A-Za-z][A-Za-z0-9_]*$")
BANNED_TYPES = (T.DecimalType,)

def enforce_gold(df, name):
    problems = []
    for fld in df.schema.fields:
        if not NAME_RE.match(fld.name):
            problems.append(f"column name {fld.name!r} would trigger Delta column mapping")
        if isinstance(fld.dataType, BANNED_TYPES):
            problems.append(f"column {fld.name!r} is {fld.dataType.simpleString()} - Fabric Graph returns null for Decimal")
    if problems:
        raise ValueError(f"{name} violates the Gold contract: " + "; ".join(problems))
    return df

TO_WRITE = {
    "dim_date":       gld_date,
    "dim_sales_org":  gld_salesorg,
    "dim_plant":      gld_plant,
    "dim_division":   gld_division,
    "dim_channel":    gld_channel,
    "dim_order_type": gld_ordertype,
    "dim_rejection":  gld_rejection,
    "fact_order_to_cash":       gld_fact,
    "fact_accounts_receivable": gld_ar,
    "fact_sales_target":        gld_target,
    "fact_document_flow":       gld_docflow,
}

for n, d in TO_WRITE.items():
    enforce_gold(d, f"{GOLD}{n}")
for n, d in [("dim_customer", gld_customer), ("dim_material", gld_material)]:
    enforce_gold(d, f"{GOLD}{n}")

print(f"Gold contract satisfied by all {len(TO_WRITE) + 2} tables (no Decimal, no exotic column names)")

## 8. Write Gold

In [ ]:
written = {}
for name, df in TO_WRITE.items():
    table = f"{GOLD}{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table))
    written[table] = spark.table(table).count()
    print(f"{table:34s}{written[table]:>10,}")

# SCD2 dimensions were already written by scd2_load - merged, never overwritten
for name in ("dim_customer", "dim_material"):
    table = f"{GOLD}{name}"
    written[table] = spark.table(table).count()
    print(f"{table:34s}{written[table]:>10,}  (SCD2, merged)")

## 9. Validation gate

In [ ]:
failures = []
def check(label, ok, detail=""):
    print(f"{'PASS' if ok else 'FAIL'}  {label}" + (f"  - {detail}" if detail else ""))
    if not ok:
        failures.append(label)

for t, n in written.items():
    check(f"{t} is non-empty", n > 0, f"{n:,} rows")

silver_lines = spark.table(f"{SILVER}order_fulfilment").count()
fact = spark.table(f"{GOLD}fact_order_to_cash")
check("fact preserves order-line grain", fact.count() == silver_lines,
      f"{fact.count():,} vs silver {silver_lines:,}")
check("OrderLineKey is unique",
      fact.select("OrderLineKey").distinct().count() == fact.count())

# the two graph rules, re-checked on what is actually on disk
for t in written:
    sch = spark.table(t).schema
    check(f"{t}: no Decimal columns",
          not any(isinstance(f.dataType, T.DecimalType) for f in sch.fields))
    check(f"{t}: all column names are graph-safe",
          all(NAME_RE.match(f.name) for f in sch.fields))

check("every fact row joins to DimDate on order date",
      fact.filter("OrderCreatedDateKey is not null")
          .join(spark.table(f"{GOLD}dim_date"), F.col("OrderCreatedDateKey") == F.col("DateKey"), "left_anti").count() == 0)

check("EUR conversion applied (EUR lines unchanged, non-EUR lines moved)",
      fact.filter("DocCurrency = 'EUR' and abs(OrderNetValueEUR - OrderNetValue) > 0.01").count() == 0
      and fact.filter("DocCurrency <> 'EUR' and OrderNetValue > 0 and OrderNetValueEUR = OrderNetValue").count() == 0)

check("no line fell back to a default FX rate", fact.filter("FxRateMissing").count() == 0,
      "a silent 1.0 fallback would misstate every non-EUR figure")

check("no negative order quantity", fact.filter("OrderQty < 0").count() == 0)
# --- process chronology --------------------------------------------------------------
# These were originally asserted at ORDER-LINE grain by comparing MAX(clearing) with
# MAX(billing) and MAX(billing) with MAX(goods issue). That is invalid, and it failed:
# comparing two independent MAX aggregates does not test a per-document ordering.
#
#   Cash "before" invoice: a line with two invoices, the first paid and the second still
#   open, gives MAX(ClearingDate) < MAX(BillingDate). Nothing is wrong. Guaranteed to occur
#   here because 612 AR items are open.
#
#   Invoice "before" goods issue: a split delivery whose second shipment is not yet invoiced
#   gives MAX(BillingDate) < MAX(GoodsIssueDate). Also correct. Guaranteed by ~12% split
#   deliveries and ~3% billing blocks.
#
# The real invariants hold at DOCUMENT grain, so that is where they are now tested. The
# line-grain counts are still printed, as information rather than as assertions.
ar_g   = spark.table(f"{GOLD}fact_accounts_receivable")
bill_s = spark.table(f"{SILVER}billing_item").filter(F.col("DeliveryId").isNotNull())
dlv_s  = (spark.table(f"{SILVER}delivery_item")
            .select("DeliveryId", "ActualGoodsIssueDate").dropDuplicates(["DeliveryId"]))

check("a receivable is never cleared before it is posted",
      ar_g.filter("ClearingDate is not null and ClearingDate < PostingDate").count() == 0,
      "AR item grain - a true invariant")

check("an invoice never predates the goods issue it bills",
      bill_s.join(dlv_s, "DeliveryId")
            .filter("BillingDate is not null and ActualGoodsIssueDate is not null "
                    "and BillingDate < ActualGoodsIssueDate").count() == 0,
      "billing-item to delivery grain - a true invariant")

_cash_lt_bill = fact.filter("LastClearingDate is not null and LastBillingDate is not null "
                            "and LastClearingDate < LastBillingDate").count()
_bill_lt_gi   = fact.filter("LastBillingDate is not null and LastGoodsIssueDate is not null "
                            "and LastBillingDate < LastGoodsIssueDate").count()
print(f"INFO  lines where MAX(clearing) < MAX(billing) : {_cash_lt_bill:,} "
      f"(expected: partly-open multi-invoice lines)")
print(f"INFO  lines where MAX(billing) < MAX(goods issue): {_bill_lt_gi:,} "
      f"(expected: partly-invoiced split deliveries)")

for d, k in [("dim_customer", "CustomerId"), ("dim_material", "MaterialId")]:
    t = spark.table(f"{GOLD}{d}")
    check(f"{d}: exactly one current version per {k}",
          t.filter("IsCurrent").groupBy(k).count().filter("count > 1").count() == 0)
    check(f"{d}: current rows are open-ended",
          t.filter(f"IsCurrent and ValidTo <> date'{HIGH_DATE}'").count() == 0)

flow_t = spark.table(f"{GOLD}fact_document_flow")
_bridge = flow_t.filter("PrecedingOrderLineKey is not null")
_nbridge = _bridge.count()
check("document-flow bridge key populated for sales-order edges", _nbridge > 0,
      f"{_nbridge:,} order-line edges")
check("every bridge key resolves to a real order line",
      _bridge.join(fact, _bridge.PrecedingOrderLineKey == fact.OrderLineKey, "left_anti").count() == 0,
      "an unresolvable bridge key would silently break graph traversal")
check("non-sales-order edges leave the bridge key NULL",
      flow_t.filter("PrecedingCategory <> 'C' and PrecedingOrderLineKey is not null").count() == 0)

ar_v = spark.table(f"{GOLD}fact_accounts_receivable")
check("no AR row fell back to a default FX rate", ar_v.filter("FxRateMissing").count() == 0,
      "a silent 1.0 would misstate every USD and SGD receivable")
check("EUR receivables are unchanged by conversion",
      ar_v.filter("Currency = 'EUR' and abs(AmountEUR - AmountLocal) > 0.01").count() == 0)
check("non-EUR receivables actually moved",
      ar_v.filter("Currency <> 'EUR' and AmountLocal > 0 and AmountEUR = AmountLocal").count() == 0)

check("sales targets cover every sales org in the fact",
      spark.table(f"{GOLD}fact_sales_target").select("SalesOrg").distinct().count()
      >= fact.select("SalesOrg").distinct().count())

print()
if failures:
    raise AssertionError(f"{len(failures)} Gold validation check(s) failed: {failures}")
print(f"All {len(written)} Gold tables written and validated.")

## 10. Headline figures for the record

In [ ]:
scope = fact.filter("not IsRejected")
fact.createOrReplaceTempView("f")

spark.sql("""
SELECT SalesOrg,
       COUNT(*)                                             AS OrderLines,
       ROUND(SUM(OrderNetValueEUR), 0)                      AS OrderedEUR,
       ROUND(SUM(BilledNetValueEUR), 0)                     AS BilledEUR,
       ROUND(AVG(CASE WHEN IsOTIF THEN 1.0 ELSE 0.0 END), 3) AS OTIF,
       ROUND(AVG(OrderToCashDays), 1)                       AS AvgOrderToCashDays
FROM   f
WHERE  NOT IsRejected
GROUP  BY SalesOrg ORDER BY SalesOrg
""").show(truncate=False)

# AR by company code, local vs group currency. The whole point of the conversion is that the
# TOTAL row is only meaningful in the EUR column - the Local column adds EUR + USD + SGD.
spark.table(f"{GOLD}fact_accounts_receivable").createOrReplaceTempView("ar")
spark.sql("""
SELECT COALESCE(CompanyCode, 'TOTAL')                  AS CompanyCode,
       MAX(Currency)                                   AS LocalCurrency,
       COUNT(*)                                        AS ArItems,
       SUM(CASE WHEN IsOpen THEN 1 ELSE 0 END)         AS OpenItems,
       ROUND(SUM(CASE WHEN IsOpen THEN SignedAmountLocal ELSE 0 END), 0) AS OpenAR_Local,
       ROUND(SUM(CASE WHEN IsOpen THEN SignedAmountEUR   ELSE 0 END), 0) AS OpenAR_EUR
FROM   ar
GROUP  BY ROLLUP(CompanyCode) ORDER BY 1
""").show(truncate=False)
print("   ^ the TOTAL row is only meaningful in the EUR column")

print("on-time vs REQUESTED :", round(scope.agg(F.avg(F.col("IsOnTime").cast("double"))).collect()[0][0], 4))
print("on-time vs CONFIRMED :", round(scope.agg(F.avg(F.col("IsOnTimeVsConfirmed").cast("double"))).collect()[0][0], 4))
print("in-full              :", round(scope.agg(F.avg(F.col("IsInFull").cast("double"))).collect()[0][0], 4))
print("OTIF                 :", round(scope.agg(F.avg(F.col("IsOTIF").cast("double"))).collect()[0][0], 4))